In [22]:
%%sql
-- Vue Temporaire 1 
CREATE OR REPLACE TEMPORARY VIEW results_golscorers AS
SELECT
    r.date,
    r.home_team,
    r.away_team,
    r.home_score,
    r.away_score,
    r.tournament,
    r.city,
    r.country,
    r.neutral,
    r.day,
    r.month,
    r.year,
    g.team as scorer_team,
    g.scorer,
    g.minute,
    g.own_goal,
    g.penalty,
    g.half
FROM Football.LH_Silver.dbo.results r
LEFT JOIN Football.LH_Silver.dbo.goalscorers g
    ON r.date = g.date
    AND r.home_team = g.home_team
    AND r.away_team = g.away_team

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 40, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [23]:
%%sql
-- Vue temporaire 2
CREATE OR REPLACE TEMPORARY VIEW facts_primary AS
SELECT 
    rg.date,
    CAST(DATE_FORMAT(rg.date, 'yyyyMMdd') AS INT) AS date_key,
    rg.home_team,
    rg.away_team,
    rg.home_score,
    rg.away_score,
    rg.tournament,
    rg.city,
    rg.country,
    rg.neutral,
    rg.day,
    rg.month,
    rg.year,
    rg.scorer_team,
    rg.scorer,
    rg.minute,
    rg.own_goal,
    rg.penalty,
    rg.half,
    s.winner,
    s.first_shooter,
    -- création des clés
    Case when rg.scorer IS NOT NULL THEN  DENSE_RANK() OVER (ORDER BY rg.scorer, rg.scorer_team) 
    Else 0 end as scorer_id,
    DENSE_RANK() OVER (ORDER BY rg.date, rg.home_team, rg.away_team) AS match_id,
    DENSE_RANK() OVER (ORDER BY rg.tournament) AS tournament_id,
    CASE 
        WHEN rg.scorer IS NOT NULL THEN 
            DENSE_RANK() OVER (ORDER BY rg.date, rg.scorer, rg.minute)
        WHEN rg.scorer IS NULL AND rg.home_score = 0 AND rg.away_score =0 THEN 0
        ELSE NULL
    END AS goal_id,
    CASE 
        WHEN s.winner IS NOT NULL THEN TRUE
        ELSE FALSE
    END AS shootouts
FROM results_golscorers rg
LEFT JOIN Football.LH_Silver.dbo.shootouts s
    ON rg.date = s.date
    AND rg.home_team = s.home_team
    AND rg.away_team = s.away_team;

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 41, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [24]:
%%sql
--Dimension date (Pas besoin de générer un calendrier)
CREATE OR REPLACE TABLE dim_date AS
SELECT DISTINCT
    date_key,
    date,
    DAY(date) AS day,
    MONTH(date) AS month,
    YEAR(date) AS year,
    CAST(FLOOR(YEAR(date) / 10) * 10 AS INT) AS decade
FROM facts_primary
WHERE date IS NOT NULL
ORDER BY date

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 42, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [25]:
%%sql
-- Dimension tournois
Create or replace Table dim_tournament as
select distinct
tournament_id,
tournament
--year,
-- city,
--country
from facts_primary

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 43, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [26]:
%%sql
-- Dimension Buteur
Create or replace Table dim_scorer as
select distinct
scorer_id,
Case 
    WHEN scorer_id = 0 THEN 'unknown'
    ELSE scorer
END AS scorer,
scorer_team
from facts_primary
-- On ne sélectionne que les buts marqués par l'équipe et pas ceux en csc par l'équipe adverse
where own_goal = 'false'

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 44, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [27]:
%%sql
-- Vue temporaire mapping nom des pays 
CREATE OR REPLACE TEMPORARY VIEW confederation_final AS
SELECT 
confederation,
CASE WHEN country ='Cabo Verde' THEN 'Cape Verde'
     WHEN country ='China' THEN 'China PR'
     WHEN country ='U.S. Virgin Islands' THEN 'United States Virgin Islands'
     ELSE country
END AS country
from confederations 

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 45, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [28]:
%%sql
-- Dimension Pays
CREATE OR REPLACE TABLE dim_country AS
WITH Country AS (
    SELECT home_team AS country FROM facts_primary
    UNION
    SELECT away_team AS country FROM facts_primary
),
-- Toutes les combinaisons pays + former_name + confederation
Country_All AS (
    SELECT
        c.country,
        f.former AS former_name,
        co.confederation
    FROM Country c
    LEFT JOIN LH_Silver.dbo.former_names f ON c.country = f.current
    LEFT JOIN confederation_final co ON c.country = co.country
),
-- Canonical : on cherche si le pays est lui-même un ancien nom
Canonical AS (
    SELECT DISTINCT
        ca.country,
        COALESCE(f.current, ca.country) AS canonical_name
    FROM Country_All ca
    LEFT JOIN LH_Silver.dbo.former_names f ON ca.country = f.former
)
SELECT
    ROW_NUMBER() OVER (ORDER BY ca.country, ca.former_name) AS country_id,
    ca.country,
    ca.former_name,
    ca.confederation,
    CASE WHEN 
        ca.confederation is not null THEN 'FIFA'
        ELSE 'Non FIFA'
    END AS fifa_status,
    cn.canonical_name,
    DENSE_RANK() OVER (ORDER BY cn.canonical_name) AS canonical_id
FROM Country_All ca
LEFT JOIN Canonical cn ON ca.country = cn.country

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 46, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [29]:
%%sql
-- Dimension Buts
CREATE OR REPLACE TABLE goals AS
SELECT DISTINCT
    f.match_id,
    f.scorer_id,
    hc.country_id AS home_country_id,
    ac.country_id AS away_country_id,
    f.goal_id,
    f.minute,
    f.own_goal,
    f.penalty,
    f.half
FROM facts_primary f
LEFT JOIN (
    SELECT country, MIN(country_id) AS country_id 
    FROM dim_country 
    GROUP BY country
) hc ON f.home_team = hc.country
LEFT JOIN (
    SELECT country, MIN(country_id) AS country_id 
    FROM dim_country 
    GROUP BY country
) ac ON f.away_team = ac.country
WHERE f.goal_id != 0 AND f.goal_id IS NOT NULL

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 47, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [30]:
%%sql
create or replace table LH_Gold.dbo.dim_date as 
Select * from LH_Silver.dbo.dim_date

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 48, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [31]:
%%sql
create or replace table LH_Gold.dbo.dim_tournament as 
Select * from LH_Silver.dbo.dim_tournament

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 49, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [32]:
%%sql
create or replace table LH_Gold.dbo.dim_scorer as 
Select * from LH_Silver.dbo.dim_scorer


StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 50, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [33]:
%%sql
create or replace table LH_Gold.dbo.dim_country as 
Select * from LH_Silver.dbo.dim_country



StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 51, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [34]:
%%sql
create or replace table LH_Gold.dbo.goals as 
Select * from LH_Silver.dbo.goals

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 52, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [35]:
%%sql
-- Dimension Matchs
CREATE OR REPLACE TABLE matchs AS
SELECT DISTINCT
--    date_id,
    date_key,
    match_id,
    tournament_id,
    home_team,
    home_score,
    away_team,
    away_score,
    home_team || ' vs ' || away_team || ' - ' || CAST(match_id AS STRING) AS match_label,
    'vs ' || away_team AS home_label,
    'vs ' || home_team AS away_label,
    neutral,
    city,
    country as Country_match,
    shootouts,
    first_shooter,
    winner
from facts_primary 
-- Filtre sur match_id 9638 car id en doublon mais résultats différent
where match_id != 9638

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 53, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [36]:
%%sql
-- Création de table pour flag 'Home' ou 'Away' chaque équpe de chaque confrontation
CREATE OR REPLACE TABLE dim_match_country AS
WITH home_matches AS (
    SELECT m.match_id, dc.country_id, 'home' AS role
    FROM matchs m
    JOIN dim_country dc ON m.home_team = dc.country
    
    UNION
    
    SELECT m.match_id, dc.country_id, 'home' AS role
    FROM matchs m
    JOIN dim_country dc ON m.home_team = dc.former_name
    WHERE dc.former_name IS NOT NULL
    AND m.home_team NOT IN (SELECT country FROM dim_country WHERE country IS NOT NULL)
),
away_matches AS (
    SELECT m.match_id, dc.country_id, 'away' AS role
    FROM matchs m
    JOIN dim_country dc ON m.away_team = dc.country
    
    UNION
    
    SELECT m.match_id, dc.country_id, 'away' AS role
    FROM matchs m
    JOIN dim_country dc ON m.away_team = dc.former_name
    WHERE dc.former_name IS NOT NULL
    AND m.away_team NOT IN (SELECT country FROM dim_country WHERE country IS NOT NULL)
)
SELECT * FROM home_matches
UNION
SELECT * FROM away_matches

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 54, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [37]:
%%sql
create or replace table LH_Gold.dbo.dim_match_country  as 
Select * from LH_Silver.dbo.dim_match_country 

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 55, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [38]:
%%sql
create or replace table LH_Gold.dbo.matchs  as 
Select * from LH_Silver.dbo.matchs

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 56, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [39]:
gold_table_path_dim_country = "abfss://Football@onelake.dfs.fabric.microsoft.com/LH_Gold.Lakehouse/Tables/dbo/dim_country"

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 57, Finished, Available, Finished, False)

In [40]:
from pyspark.sql import Row
from pyspark.sql.functions import col, concat, lit, when
# Mapping des pays avec flag_url pour récuperer les drapeaux
mapping = {
    # Europe - UEFA
    "Albania": "al", "Andorra": "ad", "Armenia": "am", "Austria": "at",
    "Azerbaijan": "az", "Belarus": "by", "Belgium": "be", "Bosnia and Herzegovina": "ba",
    "Bulgaria": "bg", "Croatia": "hr", "Cyprus": "cy", "Czechia": "cz",
    "Denmark": "dk", "England": "gb-eng", "Estonia": "ee", "Faroe Islands": "fo",
    "Finland": "fi", "France": "fr", "Georgia": "ge", "Germany": "de",
    "Gibraltar": "gi", "Greece": "gr", "Hungary": "hu", "Iceland": "is",
    "Ireland": "ie", "Israel": "il", "Italy": "it", "Kazakhstan": "kz",
    "Kosovo": "xk", "Latvia": "lv", "Liechtenstein": "li", "Lithuania": "lt",
    "Luxembourg": "lu", "Malta": "mt", "Moldova": "md", "Montenegro": "me",
    "Netherlands": "nl", "North Macedonia": "mk", "Northern Ireland": "gb-nir",
    "Norway": "no", "Poland": "pl", "Portugal": "pt", "Romania": "ro",
    "Russia": "ru", "San Marino": "sm", "Scotland": "gb-sct", "Serbia": "rs",
    "Slovakia": "sk", "Slovenia": "si", "Spain": "es", "Sweden": "se",
    "Switzerland": "ch", "Turkey": "tr", "Ukraine": "ua", "Wales": "gb-wls",

    # Afrique - CAF
    "Algeria": "dz", "Angola": "ao", "Benin": "bj", "Botswana": "bw",
    "Burkina Faso": "bf", "Burundi": "bi", "Cameroon": "cm", "Cape Verde": "cv",
    "Central African Republic": "cf", "Chad": "td", "Comoros": "km",
    "DR Congo": "cd", "Djibouti": "dj", "Egypt": "eg", "Equatorial Guinea": "gq",
    "Eritrea": "er", "Eswatini": "sz", "Ethiopia": "et", "Gabon": "ga",
    "Gambia": "gm", "Ghana": "gh", "Guinea": "gn", "Guinea-Bissau": "gw",
    "Ivory Coast": "ci", "Kenya": "ke", "Lesotho": "ls", "Liberia": "lr",
    "Libya": "ly", "Madagascar": "mg", "Malawi": "mw", "Mali": "ml",
    "Mauritania": "mr", "Mauritius": "mu", "Morocco": "ma", "Mozambique": "mz",
    "Namibia": "na", "Niger": "ne", "Nigeria": "ng", "Rwanda": "rw",
    "São Tomé and Príncipe": "st", "Senegal": "sn", "Sierra Leone": "sl",
    "Somalia": "so", "South Africa": "za", "South Sudan": "ss", "Sudan": "sd",
    "Tanzania": "tz", "Togo": "tg", "Tunisia": "tn", "Uganda": "ug",
    "Zambia": "zm", "Zimbabwe": "zw",

    # Amérique du Sud - CONMEBOL
    "Argentina": "ar", "Bolivia": "bo", "Brazil": "br", "Chile": "cl",
    "Colombia": "co", "Ecuador": "ec", "Paraguay": "py", "Peru": "pe",
    "Uruguay": "uy", "Venezuela": "ve",

    # Amérique du Nord/Centrale - CONCACAF
    "Antigua and Barbuda": "ag", "Bahamas": "bs", "Barbados": "bb",
    "Belize": "bz", "Bermuda": "bm", "Canada": "ca", "Costa Rica": "cr",
    "Cuba": "cu", "Dominica": "dm", "Dominican Republic": "do",
    "El Salvador": "sv", "Grenada": "gd", "Guatemala": "gt", "Guyana": "gy",
    "Haiti": "ht", "Honduras": "hn", "Jamaica": "jm", "Mexico": "mx",
    "Montserrat": "ms", "Nicaragua": "ni", "Panama": "pa",
    "Saint Kitts and Nevis": "kn", "Saint Lucia": "lc",
    "Saint Vincent and the Grenadines": "vc", "Suriname": "sr",
    "Trinidad and Tobago": "tt", "United States": "us",

    # Asie - AFC
    "Afghanistan": "af", "Australia": "au", "Bahrain": "bh", "Bangladesh": "bd",
    "Bhutan": "bt", "Brunei": "bn", "Cambodia": "kh", "China PR": "cn",
    "Chinese Taipei": "tw", "Guam": "gu", "Hong Kong": "hk", "India": "in",
    "Indonesia": "id", "Iran": "ir", "Iraq": "iq", "Japan": "jp",
    "Jordan": "jo", "Kuwait": "kw", "Kyrgyzstan": "kg", "Laos": "la",
    "Lebanon": "lb", "Macao": "mo", "Malaysia": "my", "Maldives": "mv",
    "Mongolia": "mn", "Myanmar": "mm", "Nepal": "np", "North Korea": "kp",
    "Oman": "om", "Pakistan": "pk", "Palestine": "ps", "Philippines": "ph",
    "Qatar": "qa", "Saudi Arabia": "sa", "Singapore": "sg", "South Korea": "kr",
    "Sri Lanka": "lk", "Syria": "sy", "Tajikistan": "tj", "Thailand": "th",
    "Timor-Leste": "tl", "Turkmenistan": "tm", "United Arab Emirates": "ae",
    "Uzbekistan": "uz", "Vietnam": "vn", "Yemen": "ye",

    # Océanie - OFC
    "American Samoa": "as", "Cook Islands": "ck", "Fiji": "fj",
    "New Caledonia": "nc", "New Zealand": "nz", "Papua New Guinea": "pg",
    "Samoa": "ws", "Solomon Islands": "sb", "Tahiti": "pf", "Tonga": "to",
    "Vanuatu": "vu",
}

# Créer le DataFrame de mapping
mapping_df = spark.createDataFrame(
    [Row(canonical_name=k, iso2_code=v) for k, v in mapping.items()]
)

# Joindre avec dim_country
dim_country = spark.read.table("dim_country")

# Supprimer iso2_code si elle existe déjà
if "iso2_code" in dim_country.columns:
    dim_country = dim_country.drop("iso2_code")
if "flag_url" in dim_country.columns:
    dim_country = dim_country.drop("flag_url")

# Jointure et ajout flag_url
dim_country_enriched = dim_country.join(mapping_df, on="canonical_name", how="left")
dim_country_enriched = dim_country_enriched.withColumn(
    "flag_url",
    when(
        col("iso2_code").isNotNull(),
        concat(lit("https://flagcdn.com/w40/"), col("iso2_code"), lit(".png"))
    ).otherwise(None)
)

# Sauvegarder
dim_country_enriched.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable("LH_Gold.dbo.dim_country")



StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 58, Finished, Available, Finished, False)

In [41]:
%%sql
DROP TABLE IF EXISTS LH_Silver.dbo.matchs;
DROP TABLE IF EXISTS LH_Silver.dbo.goals;
DROP TABLE IF EXISTS LH_Silver.dbo.dim_match_country ;
DROP TABLE IF EXISTS LH_Silver.dbo.dim_country;
DROP TABLE IF EXISTS LH_Silver.dbo.dim_scorer;
DROP TABLE IF EXISTS LH_Silver.dbo.dim_date;
DROP TABLE IF EXISTS LH_Silver.dbo.dim_tournament;
DROP TABLE IF EXISTS LH_Silver.dbo.facts;
DROP TABLE IF EXISTS LH_Gold.dbo.facts;
DROP TABLE IF EXISTS LH_Gold.dbo.dim_match;

StatementMeta(, f43c5e4e-0827-4d37-8e6f-9795e36ebd06, 68, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

...

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>